# LIBERO Simulation Benchmark Evaluation

Evaluates a trained `TransformerFlowMatchingPolicy` on the LIBERO benchmark.

**Metric**: Task success rate (%) — fraction of rollouts where the robot completes the task within `max_steps`.

**Standard protocol**: 50 rollouts × 10 tasks per suite, reported per suite and as overall average.

**Suites**: `libero_spatial`, `libero_object`, `libero_goal`, `libero_long`

## 1. Install Dependencies

In [ ]:
# System deps for headless rendering
!apt-get install -y xvfb libgl1-mesa-glx libegl1-mesa libgles2-mesa-dev 2>/dev/null

# Python deps
!pip install -q mujoco pyvirtualdisplay
!pip install -q robosuite

# LIBERO benchmark library
!pip install -q git+https://github.com/Lifelong-Robot-Learning/LIBERO.git

# LeRobot and repo deps
!pip install -q lerobot safetensors huggingface_hub

## 2. Setup Virtual Display (Colab Headless Rendering)

In [ ]:
import os
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

# Tell MuJoCo to use EGL (GPU-accelerated headless rendering on Colab)
os.environ["MUJOCO_GL"] = "egl"
os.environ["DISPLAY"] = ":99"

print("Virtual display started.")

## 3. Clone Repo and Setup Path

In [ ]:
import sys
from pathlib import Path

# Clone the repo if not already present
if not Path("lerobot-piper").exists():
    !git clone https://github.com/mayuanyang/lerobot-piper.git

repo_src = Path("lerobot-piper/src")
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

print(f"Python path includes: {repo_src}")

## 4. Imports

In [ ]:
import torch
import numpy as np
from collections import deque
from pathlib import Path
import pandas as pd
import huggingface_hub

from lerobot.policies.factory import load_pre_post_processors
from models.transformer_flow_matching.transformer_flow_matching_policy import TransformerFlowMatchingPolicy

from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 5. Configuration

In [ ]:
# ── Checkpoint ────────────────────────────────────────────────────────────────
# HuggingFace repo ID  OR  local path to a checkpoint directory
CHECKPOINT = "ISdept/your-libero-checkpoint"   # <-- change this

# ── Evaluation settings ───────────────────────────────────────────────────────
# Suites to evaluate. Comment out any you want to skip.
SUITES = [
    "libero_spatial",
    "libero_object",
    "libero_goal",
    "libero_long",
]

NUM_ROLLOUTS = 50    # rollouts per task (paper standard = 50)
MAX_STEPS    = 300   # max env steps per rollout
IMAGE_SIZE   = 256   # must match training resolution

# ── Action chunking ───────────────────────────────────────────────────────────
# How many predicted actions to execute before re-planning.
# Smaller = more reactive but slower; larger = faster but less reactive.
# Set to 1 to re-plan every step (most conservative).
N_ACTION_STEPS_TO_EXECUTE = 8

print(f"Checkpoint : {CHECKPOINT}")
print(f"Suites     : {SUITES}")
print(f"Rollouts   : {NUM_ROLLOUTS} per task")
print(f"Max steps  : {MAX_STEPS}")
print(f"Action chunk: {N_ACTION_STEPS_TO_EXECUTE} steps before re-plan")

## 6. Load Policy

In [ ]:
ckpt_path = Path(CHECKPOINT)
if not ckpt_path.exists():
    print(f"Downloading checkpoint from HuggingFace: {CHECKPOINT}")
    ckpt_path = Path(huggingface_hub.snapshot_download(CHECKPOINT))

print(f"Loading policy from: {ckpt_path}")
policy = TransformerFlowMatchingPolicy.from_pretrained(ckpt_path)
policy.eval()
policy.to(device)

preprocessor, postprocessor = load_pre_post_processors(ckpt_path)
if isinstance(preprocessor, torch.nn.Module):
    preprocessor.to(device)

n_obs_steps = policy.config.n_obs_steps
horizon      = policy.config.horizon
action_dim   = policy.config.action_dim
state_dim    = policy.config.state_dim

print(f"Policy loaded.")
print(f"  n_obs_steps : {n_obs_steps}")
print(f"  horizon     : {horizon}")
print(f"  action_dim  : {action_dim}")
print(f"  state_dim   : {state_dim}")

## 7. Inspect Environment Observation Keys (Debug)

Run this once to verify which LIBERO observation keys map to your policy inputs.

In [ ]:
benchmark_dict = benchmark.get_benchmark_dict()
task_suite     = benchmark_dict["libero_goal"]()
task           = task_suite.get_task(0)
bddl_file      = task_suite.get_task_bddl_file_path(0)

env = OffScreenRenderEnv(**{
    "bddl_file_name": bddl_file,
    "camera_heights": IMAGE_SIZE,
    "camera_widths":  IMAGE_SIZE,
})
obs = env.reset()
env.close()

print("LIBERO env observation keys and shapes:")
for k, v in obs.items():
    v = np.array(v)
    print(f"  {k:40s}  shape={v.shape}  dtype={v.dtype}")

print(f"\nPolicy expects:")
print(f"  observation.state              shape=({n_obs_steps}, {state_dim})")
print(f"  observation.images.image       shape=(3, {IMAGE_SIZE}, {IMAGE_SIZE})")
print(f"  observation.images.image2      shape=(3, {IMAGE_SIZE}, {IMAGE_SIZE})")
print(f"  action                         shape=({action_dim},)")

## 8. Observation Preprocessing Utilities

Maps LIBERO env obs → policy input batch.

**Verify the `build_state` mapping against the debug output above.**
The default assumes `robot0_joint_pos` (7-dim) + one gripper scalar = 8-dim.
Adjust if your `state_dim` or obs keys differ.

In [ ]:
def img_to_tensor(img: np.ndarray) -> torch.Tensor:
    """(H, W, C) uint8 → (C, H, W) float32 in [0, 1]."""
    t = torch.from_numpy(img.copy()).float() / 255.0
    return t.permute(2, 0, 1)  # CHW


def build_state(obs: dict) -> np.ndarray:
    """
    Build the proprioceptive state vector from a LIBERO observation dict.

    Default mapping (verify against debug cell output above):
      robot0_joint_pos   (7,)  -- arm joint positions
      robot0_gripper_qpos (2,) -- gripper finger positions → take mean for 1 scalar
      → state (8,)

    If state_dim is different, adjust the concatenation here.
    """
    joint_pos = np.array(obs["robot0_joint_pos"])           # (7,)
    gripper   = np.array(obs["robot0_gripper_qpos"])        # (2,)
    return np.concatenate([joint_pos, gripper[:1]])          # (8,)


def build_batch(obs_buffer: deque, task_description: str) -> dict:
    """
    Convert a deque of recent observations into a batch dict for the policy.

    obs_buffer entries are dicts with keys: state, image, image2.
    The policy expects the temporal dimension (n_obs_steps) pre-stacked for state,
    and only the current (last) frame for images.
    """
    # State: (1, n_obs_steps, state_dim)
    states = torch.stack([torch.tensor(o["state"], dtype=torch.float32)
                          for o in obs_buffer]).unsqueeze(0).to(device)

    # Images: (1, C, H, W) — current frame only
    img  = obs_buffer[-1]["image"].unsqueeze(0).to(device)
    img2 = obs_buffer[-1]["image2"].unsqueeze(0).to(device)

    return {
        "observation.state":          states,
        "observation.images.image":   img,
        "observation.images.image2":  img2,
        "task_description":           [task_description],
    }


print("Preprocessing utilities defined.")

## 9. Single Episode Evaluation

In [ ]:
def run_episode(env, task_description: str, seed: int) -> bool:
    """
    Run one rollout. Returns True if the task succeeds.

    Action chunking: the policy predicts `horizon` actions at once.
    We execute N_ACTION_STEPS_TO_EXECUTE of them before re-planning.
    """
    obs = env.reset()
    env.env.seed(seed)

    # Observation buffer — padded with the first obs until full
    first_frame = {
        "state":  build_state(obs),
        "image":  img_to_tensor(obs["agentview_image"]),
        "image2": img_to_tensor(obs["robot0_eye_in_hand_image"]),
    }
    obs_buffer = deque(
        [first_frame] * n_obs_steps,
        maxlen=n_obs_steps,
    )

    action_queue = deque()   # pre-computed actions waiting to be executed

    for step in range(MAX_STEPS):
        # Re-plan when the action queue is empty
        if not action_queue:
            batch = build_batch(obs_buffer, task_description)
            batch = preprocessor(batch)

            with torch.no_grad():
                # select_action returns (1, horizon, action_dim) or (1, action_dim)
                actions = policy.select_action(batch)

            if actions.dim() == 3:
                # Chunk: queue up the first N_ACTION_STEPS_TO_EXECUTE actions
                chunk = actions[0, :N_ACTION_STEPS_TO_EXECUTE].cpu().numpy()
            else:
                # Single action
                chunk = actions[0].unsqueeze(0).cpu().numpy()

            action_queue.extend(chunk)

        action = action_queue.popleft()   # (action_dim,)

        obs, reward, done, info = env.step(action)

        # Update observation buffer with new frame
        obs_buffer.append({
            "state":  build_state(obs),
            "image":  img_to_tensor(obs["agentview_image"]),
            "image2": img_to_tensor(obs["robot0_eye_in_hand_image"]),
        })

        if done:
            return True

    return False  # timed out


print("Episode evaluation function defined.")

## 10. Suite Evaluation

In [ ]:
def evaluate_suite(suite_name: str) -> dict:
    """
    Evaluate all tasks in a LIBERO suite.
    Returns a dict with per-task and overall success rates.
    """
    print(f"\n{'='*60}")
    print(f"Suite: {suite_name}")
    print(f"{'='*60}")

    benchmark_dict = benchmark.get_benchmark_dict()
    if suite_name not in benchmark_dict:
        raise ValueError(f"Unknown suite '{suite_name}'. "
                         f"Available: {list(benchmark_dict.keys())}")

    task_suite  = benchmark_dict[suite_name]()
    num_tasks   = task_suite.get_num_tasks()
    results     = {}

    for task_id in range(num_tasks):
        task        = task_suite.get_task(task_id)
        task_name   = task.name
        bddl_file   = task_suite.get_task_bddl_file_path(task_id)
        task_desc   = task.language_instruction

        env = OffScreenRenderEnv(**{
            "bddl_file_name": bddl_file,
            "camera_heights": IMAGE_SIZE,
            "camera_widths":  IMAGE_SIZE,
        })

        successes = 0
        for rollout_idx in range(NUM_ROLLOUTS):
            success = run_episode(env, task_desc, seed=rollout_idx)
            successes += int(success)
            print(f"  Task {task_id:2d}/{num_tasks-1}  rollout {rollout_idx+1:3d}/{NUM_ROLLOUTS}  "
                  f"success={'✓' if success else '✗'}  "
                  f"running={successes/(rollout_idx+1)*100:.1f}%",
                  end="\r")

        env.close()

        rate = successes / NUM_ROLLOUTS * 100
        results[task_name] = {"successes": successes, "total": NUM_ROLLOUTS, "rate": rate}
        print(f"  Task {task_id:2d} [{task_name}]  → {rate:.1f}%  ({successes}/{NUM_ROLLOUTS})")

    suite_rate = np.mean([v["rate"] for v in results.values()])
    print(f"\n  {suite_name} average: {suite_rate:.1f}%")
    return {"tasks": results, "suite_avg": suite_rate}


print("Suite evaluation function defined.")

## 11. Run Evaluation

In [ ]:
all_results = {}

for suite in SUITES:
    try:
        all_results[suite] = evaluate_suite(suite)
    except Exception as e:
        print(f"\nSkipped {suite}: {e}")

print("\n\nEvaluation complete.")

## 12. Results

In [ ]:
# Per-suite summary table
rows = []
for suite, data in all_results.items():
    for task_name, task_data in data["tasks"].items():
        rows.append({
            "Suite":    suite,
            "Task":     task_name,
            "Success": f"{task_data['successes']}/{task_data['total']}",
            "Rate (%)": f"{task_data['rate']:.1f}",
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Overall summary
print("\n" + "="*50)
print("Suite Averages:")
suite_avgs = []
for suite, data in all_results.items():
    avg = data["suite_avg"]
    suite_avgs.append(avg)
    print(f"  {suite:20s}  {avg:5.1f}%")

if suite_avgs:
    overall = np.mean(suite_avgs)
    print(f"  {'Overall avg':20s}  {overall:5.1f}%")